## Notebook for Experimenting Summarizer Generation

Author : Bagas Noor Fadhilah  
Student Number : 18223115  
  
---  
This notebook serves as an experiment playground for finetuning a model for summarization purpose and is part of a 3 stage pipeline. This model would be on the end of the pipeline or presumably the 3rd stage. The planned metrics that are going to be used are ROUGE, NLI, and if possible BERTScore. ROUGE serves as a lexical metric that is similar to EM (Exact Matching) and NLI would be monitoring the hallucinations that could be produced by the model. The model is going to be deployed via ngrok using their endpoint and the endpoint itself is going to be use from our local laptop that runs the python CLI script for demo.
  



```
{
  "input_text": "Summarize the following Steam reviews into a concise paragraph. Adhere strictly to the target aspects and their sentiments.\n\nTarget Aspects: Content: vast (positive) | Progression: rewarding (positive) | UI: dated (negative)\n\nReviews:\n- There is an insane amount of bosses, biomes, and gear to craft.\n- Progression feels earned every time you take down hardmode bosses.\n- Inventory management and UI feel cluttered and old-school.",
  "target_text": "Players praise the vast amount of craftable content and rewarding progression system, though the inventory UI feels dated and cluttered."
}
```

In [ ]:
!pip install evaluate
import evaluate

!pip install rouge_score



In [9]:
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import gc
import os

from datasets import load_dataset, Dataset, DatasetDict
from evaluate import load
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, PeftModel
from pathlib import Path

In [ ]:
SEED = 42
FLAN_MODEL = "google/flan-t5-base"
BART_MODEL = "facebook/bart-base"
BART_MODEL = "facebook/bart-large-cnn"
NLI_MODEL = "cross-encoder/nli-deberta-v3-base"
set_seed(42)

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


### Initial Dataset Loading

In [ ]:
sample = {
  "input_text": "Summarize the following Steam reviews into a concise paragraph. Adhere strictly to the target aspects and their sentiments.\n\nTarget Aspects: Content: vast (positive) | Progression: rewarding (positive) | UI: dated (negative)\n\nReviews:\n- There is an insane amount of bosses, biomes, and gear to craft.\n- Progression feels earned every time you take down hardmode bosses.\n- Inventory management and UI feel cluttered and old-school.",
  "target_text": "Players praise the vast amount of craftable content and rewarding progression system, though the inventory UI feels dated and cluttered."
}


### Utility Functions

In [ ]:
FLAN_tokenizer = AutoTokenizer.from_pretrained(FLAN_MODEL)
BART_tokenizer = AutoTokenizer.from_pretrained(BART_MODEL)
NLI_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [ ]:
tokenized_sample = BART_tokenizer(sample['input_text'])

In [ ]:
tokenized_sample

{'input_ids': [0, 38182, 3916, 2072, 5, 511, 18120, 6173, 88, 10, 6561, 17818, 4, 1614, 10859, 14657, 7, 5, 1002, 5894, 8, 49, 18316, 4, 50118, 50118, 41858, 287, 31084, 35, 12803, 35, 4714, 36, 22173, 43, 1721, 1698, 42223, 35, 16044, 36, 22173, 43, 1721, 27182, 35, 7000, 36, 33407, 43, 50118, 50118, 32773, 29, 35, 50118, 12, 345, 16, 41, 18544, 1280, 9, 14063, 6, 4003, 9898, 6, 8, 6351, 7, 6306, 4, 50118, 12, 1698, 42223, 2653, 2208, 358, 86, 47, 185, 159, 543, 42253, 14063, 4, 50118, 12, 40829, 1052, 8, 27182, 619, 29409, 10001, 8, 793, 12, 8813, 4, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
BART_tokenizer.decode(tokenized_sample['input_ids'])

'<s>Summarize the following Steam reviews into a concise paragraph. Adhere strictly to the target aspects and their sentiments.\n\nTarget Aspects: Content: vast (positive) | Progression: rewarding (positive) | UI: dated (negative)\n\nReviews:\n- There is an insane amount of bosses, biomes, and gear to craft.\n- Progression feels earned every time you take down hardmode bosses.\n- Inventory management and UI feel cluttered and old-school.</s>'

In [ ]:
# max_length = 512

def tok_func(batch, tokenizer: AutoTokenizer):
  context = tokenizer(
      text=batch["input_text"],
      truncation=True,
      padding=False,
      # max_length=max_length
  )
  summary = tokenizer(
      text=batch["target_text"],
      truncation=True,
      padding=False
  )
  context['labels'] = summary['input_ids']
  return context

In [ ]:
rouge_metric = evaluate.load("rouge")

evaluation_sample_premise = "There is an insane amount of bosses, biomes, and gear to craft. Progression feels earned every time you take down hardmode bosses. Inventory management and UI feel cluttered and old-school."

def compute_metric_FLAN(eval_pred):
  predictions, labels = eval_pred

  decoded_preds = FLAN_tokenizer.batch_decode(predictions, skip_special_tokens=True)
  labels = np.where(labels != -100, labels, FLAN_tokenizer.pad_token_id)
  decoded_labels = FLAN_tokenizer.batch_decode(labels, skip_special_tokens=True)

  result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
  metrics = {k: round(v * 100, 4) for k, v in result.items()}

  return metrics

In [ ]:
# NLI Evaluation Script

# NLI_model = CrossEncoder(NLI_MODEL)

# premises = [evaluation_sample_premise] # Baru Placeholder
# NLI_scores = []

# for premise, hypothesis in zip(premises, decoded_labels):
#   if not decoded_labels:
#     NLI_scores.append(0.0)

#   input = NLI_tokenizer(
#       premise, hypothesis, truncation=True, return_tensors='pt'
#   )

#   with torch.no_grad:
#     scores = NLI_model(**input).logits
#     probs = torch.softmax(scores, dims=-1).cpu().numpy() # target the class dims
#     p_entail = probs[1] # [contradiction, entailment, neutral]
#     NLI_scores.append(p_entail)

# metrics['Entailment'] = round(float(np.mean(NLI_scores)), 4)

### Dataset Preps

In [ ]:
sample_train_ds = pd.DataFrame(sample, index=[0])

In [ ]:
train_ds = Dataset.from_pandas(sample_train_ds)

In [ ]:
tokenized_flan_dataset = DatasetDict({
    "train": train_ds.map(lambda b: tok_func(b, FLAN_tokenizer), batched=True, desc="Tokenizing train")
})

Tokenizing train:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
data_collator_flan = DataCollatorForSeq2Seq(tokenizer=FLAN_tokenizer)

In [ ]:
tokenized_flan_dataset['train']

Dataset({
    features: ['input_text', 'target_text', '__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 1
})

### Training

In [ ]:
FLAN_model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_MODEL)
# BART_model = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL)
# BART_CNN_model = AutoModelForSeq2SeqLM.from_pretrained(BART_CNN_MODEL)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
FLAN_training_args = Seq2SeqTrainingArguments(
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    # gradient_checkpointing=True,
    num_train_epochs=1,
    weight_decay=0.01,
    warmup_steps=0,
    predict_with_generate=True,
    seed=SEED
)

In [ ]:
trainer_FLAN = Seq2SeqTrainer(
    FLAN_model,
    args=FLAN_training_args,
    train_dataset=tokenized_flan_dataset['train'],
    eval_dataset=tokenized_flan_dataset['train'], # Placeholder
    data_collator=data_collator_flan,
    compute_metrics=compute_metric_FLAN
)

In [ ]:
trainer_FLAN.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1, training_loss=2.5811355113983154, metrics={'train_runtime': 156.563, 'train_samples_per_second': 0.006, 'train_steps_per_second': 0.006, 'total_flos': 145778420736.0, 'train_loss': 2.5811355113983154, 'epoch': 1.0})

In [ ]:
trainer_FLAN.evaluate()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Rouge1,Rouge2,Rougel,Rougelsum
No log,2.477219,0,40.000000,12.121200,34.285700,34.285700


{'eval_loss': 2.4772186279296875,
 'eval_rouge1': 40.0,
 'eval_rouge2': 12.1212,
 'eval_rougeL': 34.2857,
 'eval_rougeLsum': 34.2857}

### Evaluations

In [ ]:
output = FLAN_model.generate(**FLAN_tokenizer(sample['input_text'], return_tensors='pt'))

print(FLAN_tokenizer.decode(output[0]))

<pad> The game is dated and dated, but it's still a lot of fun.


In [ ]:
output = trainer_FLAN.model.generate(**FLAN_tokenizer(sample['input_text'], return_tensors='pt'))

print(FLAN_tokenizer.decode(output[0], skip_special_tokens=True))

The


### Model Deployment

In [1]:
!pip -q install pyngrok
from pyngrok import ngrok

In [2]:
import time
import threading
from werkzeug.serving import make_server
from threading import Event
from flask import Flask, request, jsonify

In [3]:
from google.colab import userdata

In [27]:
# Clear ngrok endpoint usage
ngrok.kill()

In [23]:
app = Flask(__name__)
port = 5000

public_url = ngrok.connect(port).public_url
app.config["BASE_URL"] = public_url

@app.route("/")
def index():
  return "Hello from Colab phase 3!"

@app.route("/inference", methods=["POST"])
def inference():
  return jsonify({"text": "not implemented"})

class ThreadingServer(threading.Thread):
  def __init__(self, app, port):
    super().__init__()
    self.server = make_server("127.0.0.1", port, app)

  def run(self):
    print("Starting server")
    self.server.serve_forever()

  def shutdown(self):
    print("Shutting down server")
    self.server.shutdown()


In [6]:
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{port}\"")

 * ngrok tunnel "https://ongoing-vacant-savor.ngrok-free.dev" -> "http://127.0.0.1:5000"


In [25]:
server_thread = ThreadingServer(app, 5000)
server_thread.start() # Create new process in OS level so no blocking

Starting server


In [26]:
# Scram button for shutting down the server
server_thread.shutdown()
server_thread.join()

Shutting down server


### Github Commands

In [29]:
!git clone https://github.com/RinMDfa-a/IF5153_NLP_Final_Projects.git

Cloning into 'IF5153_NLP_Final_Projects'...
fatal: could not read Username for 'https://github.com': No such device or address
